In [2]:
import pandas as pd
import numpy as np

In [3]:
FILEPATH = "../Raw/dataset_2026-04-29T09_37_00.040908035Z_DEFAULT_INTEGRATION_IMF.STA_PIP_5.0.0.csv"

In [4]:
# Get size of file with terminal
!ls -lh {FILEPATH}

-rw-r--r--@ 1 jesper  staff    11G Apr 29 11:48 ../Raw/dataset_2026-04-29T09_37_00.040908035Z_DEFAULT_INTEGRATION_IMF.STA_PIP_5.0.0.csv


In [5]:
df = pd.read_csv(FILEPATH, low_memory=True)
df

/var/folders/_x/74827dzs033cwdj2j4gch59r0000gn/T/ipykernel_5850/3169877581.py:1: DtypeWarning: Columns (14,15,16,18,43) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(FILEPATH, low_memory=True)


,DATASET,SERIES_CODE,OBS_MEASURE,COUNTRY,ACCOUNTING_ENTRY,INDICATOR,SECTOR,COUNTERPART_SECTOR,COUNTERPART_COUNTRY,FREQUENCY,...,2022,2022-S1,2022-S2,2023,2023-S1,2023-S2,2024,2024-S1,2024-S2,2025-S1
0,IMF.STA:PIP(5.0.0),CWX.A.P_F3_L_P_USD.S121.S1.ATG.A,OBS_VALUE,Curaçao and Sint Maarten,Assets,"Portfolio investment, Debt securities, Long-te...",Central bank,Total economy,Antigua and Barbuda,Annual,...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,IMF.STA:PIP(5.0.0),CWX.A.P_F3_L_P_USD.S121.S1.ISL.A,OBS_VALUE,Curaçao and Sint Maarten,Assets,"Portfolio investment, Debt securities, Long-te...",Central bank,Total economy,Iceland,Annual,...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,IMF.STA:PIP(5.0.0),CWX.A.P_F3_L_P_USD.S121.S1.BHS.A,OBS_VALUE,Curaçao and Sint Maarten,Assets,"Portfolio investment, Debt securities, Long-te...",Central bank,Total economy,"Bahamas, The",Annual,...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,IMF.STA:PIP(5.0.0),CWX.A.P_F3_L_P_USD.S121.S1.DZA.A,OBS_VALUE,Curaçao and Sint Maarten,Assets,"Portfolio investment, Debt securities, Long-te...",Central bank,Total economy,Algeria,Annual,...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,IMF.STA:PIP(5.0.0),CWX.A.P_F3_L_P_USD.S121.S1.FRA.A,OBS_VALUE,Curaçao and Sint Maarten,Assets,"Portfolio investment, Debt securities, Long-te...",Central bank,Total economy,France,Annual,...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3140577,IMF.STA:PIP(5.0.0),ROU.L.P_F51_P_SCC_USD.S1.S1.NZL.S,OBS_VALUE,Romania,Liabilities,"Portfolio investment, Equity, Positions, Deriv...",Total economy,Total economy,New Zealand,"Half-yearly, semester",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3140578,IMF.STA:PIP(5.0.0),ROU.L.P_TOTINV_P_SCC_USD.S1.S1.ZAF.S,OBS_VALUE,Romania,Liabilities,"Portfolio investment, Total investment, Positi...",Total economy,Total economy,South Africa,"Half-yearly, semester",...,NaN,0.794509,0.941534,NaN,1.05503,0.000000,NaN,2.197700,2.027319,2.249668
3140579,IMF.STA:PIP(5.0.0),ROU.L.P_F51_P_SCC_USD.S1.S1.ARG.S,OBS_VALUE,Romania,Liabilities,"Portfolio investment, Equity, Positions, Deriv...",Total economy,Total economy,Argentina,"Half-yearly, semester",...,NaN,0.000000,0.000000,NaN,0.00000,0.000000,NaN,0.000000,0.000000,0.000000
3140580,IMF.STA:PIP(5.0.0),ROU.L.P_TOTINV_P_SCC_USD.S1.S1.BRA.S,OBS_VALUE,Romania,Liabilities,"Portfolio investment, Total investment, Positi...",Total economy,Total economy,Brazil,"Half-yearly, semester",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.205699,0.232956


In [6]:
# Annual
yr_cols = [c for c in df.columns if (c.startswith("20") or c.startswith("19")) and len(c) == 4]
# Semi-annual
se_cols = [c for c in df.columns if (c.startswith("20") or c.startswith("19")) and len(c) == 7]
# Other columns
other_cols = [c for c in df.columns if c not in yr_cols + se_cols]

In [7]:
df[df["COUNTERPART_COUNTRY"].str.contains("world", case=False, regex=False)]["COUNTERPART_COUNTRY"].value_counts()

COUNTERPART_COUNTRY
World                                           21881
World Minus 25 Significant Financial Centers     8604
Name: count, dtype: int64

In [17]:
df = df[df["FREQUENCY"] == "Annual"].copy()
df = df[df["SECTOR"] == "Total economy"].copy()
df = df[df["COUNTERPART_SECTOR"] == "Total economy"].copy()
df = df[df["ACCOUNTING_ENTRY"] == "Assets"].copy()
df = df[df["INDICATOR"] == "Portfolio investment, Total investment, Positions"].copy()

In [18]:
df_world = df[df["COUNTERPART_COUNTRY"] == "World"].copy()
df_world.shape

(94, 130)

In [19]:
df_bil = df[~df["COUNTERPART_COUNTRY"].str.contains("World")].copy()
df_bil.shape

(21903, 130)

In [23]:
# Map to iso3
EUROSTAT_COUNTRY_TO_ISO3 = {
    "Austria": "AUT",
    "Belgium": "BEL",
    "Bulgaria": "BGR",
    "Croatia": "HRV",
    "Cyprus": "CYP",
    "Czechia": "CZE",
    "Czech Republic": "CZE",
    "Denmark": "DNK",
    "Estonia": "EST",
    "Estonia, Republic of": "EST",
    "Finland": "FIN",
    "France": "FRA",
    "Germany": "DEU",
    "Greece": "GRC",
    "Hungary": "HUN",
    "Ireland": "IRL",
    "Italy": "ITA",
    "Latvia": "LVA",
    "Latvia, Republic of": "LVA",
    "Lithuania": "LTU",
    "Lithuania, Republic of": "LTU",
    "Luxembourg": "LUX",
    "Malta": "MLT",
    "Netherlands": "NLD",
    "Netherlands, The": "NLD",
    "Poland": "POL",
    "Poland, Republic of": "POL",
    "Portugal": "PRT",
    "Romania": "ROU",
    "Slovakia": "SVK",
    "Slovak Republic": "SVK",
    "Slovenia": "SVN",
    "Slovenia, Republic of": "SVN",
    "Spain": "ESP",
    "Sweden": "SWE",
    "United Kingdom": "GBR",
    "Norway": "NOR",
    "Switzerland": "CHE",
    "United States": "USA"
}

df_world["iso3"] = df_world["COUNTRY"].map(EUROSTAT_COUNTRY_TO_ISO3)

df_bil["iso3_i"] = df_bil["COUNTRY"].map(EUROSTAT_COUNTRY_TO_ISO3)
df_bil["iso3_j"] = df_bil["COUNTERPART_COUNTRY"].map(EUROSTAT_COUNTRY_TO_ISO3)

print(df_world["iso3"].isna().mean())
print(df_bil["iso3_i"].isna().mean())
print(df_bil["iso3_j"].isna().mean())

# Drop nan iso3 rows
df_world = df_world.dropna(subset=["iso3"])
df_bil = df_bil.dropna(subset=["iso3_i", "iso3_j"])

print(df_world.shape)
print(df_bil.shape)

0.0
0.0
0.0
(30, 131)
(870, 132)


In [25]:
# Melt into long format
df_world_long = df_world.melt(
    id_vars=other_cols + ["iso3"],
    value_vars=yr_cols,
    var_name="year",
    value_name="value",
)

df_bil_long = df_bil.melt(
    id_vars=other_cols + ["iso3_i", "iso3_j"],
    value_vars=yr_cols,
    var_name="year",
    value_name="value",
)

# Only save the [iso3, year, value] columns
df_world_long = df_world_long[["iso3", "year", "value"]]
df_bil_long = df_bil_long[["iso3_i", "iso3_j", "year", "value"]]

display(df_world_long.head(3))
display(df_bil_long.head(3))

,iso3,year,value
0,DNK,1997,43096.249634
1,LUX,1997,NaN
2,MLT,1997,NaN


,iso3_i,iso3_j,year,value
0,DNK,BEL,1997,181.951362
1,DNK,BGR,1997,0.000000
2,DNK,AUT,1997,243.920305


In [26]:
# Save to csv
df_world_long.to_csv("../Clean/IMF_PIP_total.csv", index=False)
df_bil_long.to_csv("../Clean/IMF_PIP_bil.csv", index=False)

In [16]:
# Search with re for a coutnry
import re
search_term = "croa"

matches = [name for name in df["COUNTRY"].unique() if re.search(search_term, name, re.IGNORECASE)]
print(matches)

if len(matches) == 1:
    sample_rows = df[df["COUNTRY"] == matches[0]]
    sample_rows = sample_rows[sample_rows["ACCOUNTING_ENTRY"] == "Assets"]
    print(sample_rows)

[]
